# PyJST CPU/GPU performance demo

Select a Colab GPU runtime, then run this notebook to create a high-resolution four-series CPU/CuPy scaling plot and downloadable raw metrics.

In [ ]:
# Install the current PyJST source and optional plotting/CUDA dependencies.
%pip install -q --upgrade --force-reinstall "git+https://github.com/VishalKandala/PyJST.git" "matplotlib>=3.7" "cupy-cuda12x>=13.0"


In [ ]:
from pathlib import Path
import importlib
import json
import shutil

import cupy as cp
import pyjst.performance as performance
importlib.reload(performance)
from pyjst.performance import benchmark_suite, save_benchmark_figure, write_benchmark_results

device = cp.cuda.runtime.getDeviceProperties(cp.cuda.Device().id)["name"].decode()
print(f"CuPy {cp.__version__} on {device}")
output_dir = Path("pyjst-performance-results")
output_dir.mkdir(exist_ok=True)


In [ ]:
# Both backends execute in this one GPU-runtime session.
# GPU timings synchronize CUDA and exclude one-time setup/transfer cost.
resolutions = (64, 128, 256, 512, 1024, 2048)
cpu_samples = benchmark_suite(resolutions, repeats=2, warmup_iterations=1, backend="numpy")
gpu_samples = benchmark_suite(resolutions, repeats=3, warmup_iterations=1, backend="cupy")
samples = cpu_samples + gpu_samples

benchmark_json = output_dir / "cpu-gpu-scaling.json"
benchmark_png = output_dir / "cpu-gpu-scaling.png"
write_benchmark_results(samples, benchmark_json)
save_benchmark_figure(samples, benchmark_png, dpi=300)
(output_dir / "run-metadata.json").write_text(json.dumps({
    "gpu": device, "cupy": cp.__version__,
    "cpu_repeats": 2, "gpu_repeats": 3, "warmup_iterations": 1,
}, indent=2) + "\n")

from IPython.display import Image, display
display(Image(filename=str(benchmark_png)))
archive = shutil.make_archive("pyjst-performance-results", "zip", root_dir=output_dir)
from google.colab import files
files.download(archive)
